# Fine-tuning QLoRA avec Unsloth : extraction JSON de factures

Ce notebook entraîne un modèle open-weights (par défaut Qwen2.5-3B-Instruct) en QLoRA sur `data/train.jsonl` et `data/val.jsonl`, au format ChatML.

**Prérequis**
- Colab : activer Exécution > Modifier le type d'exécution > GPU T4. Kaggle : activer GPU et Internet dans Settings.
- Charger les fichiers v3 locaux `data/train.jsonl` (480 lignes) et `data/val.jsonl` (60 lignes). Un clone ancien du dépôt ne contient pas nécessairement ces données non publiées.
- Colab : téléverser les deux JSONL dans `/content/data/` (ou `/content/`). Kaggle : joindre ces fichiers comme dataset privé, puis régler `DATA_DIR_OVERRIDE` si plusieurs copies existent.
- Exécuter les cellules dans l'ordre. Si l'installation demande un redémarrage du runtime, le redémarrer avant de continuer.
- Les PNG ne sont pas utilisés : entraînement textuel uniquement, sans OCR. Aucun upload Hub/W&B par défaut.
- Les poids LoRA, le tokenizer et la provenance seront enregistrés dans `results/<RUN_NAME>/`, puis archivés pour téléchargement. Aucun entraînement n'a été exécuté lors de la préparation de ce notebook.

In [ ]:
%pip install -q unsloth trl datasets matplotlib

## 1. Configuration

In [ ]:
import json
import os
import hashlib
import shutil
from collections import Counter
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct'  # alternative : 'unsloth/Llama-3.2-3B-Instruct'
MAX_SEQ_LENGTH = 4096
LOAD_IN_4BIT = True
LORA_R = 16
LORA_ALPHA = 16
SEED = 42
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
BATCH_SIZE = 1
GRAD_ACCUM = 8
USE_WANDB = False  # True : journalise la perte dans Weights & Biases (WANDB_API_KEY requis)
HF_REPO_GGUF = None  # ex. 'user/invoice-extractor-gguf' pour pousser le GGUF sur le Hub
DATA_DIR_OVERRIDE = None  # ex. Path('/kaggle/input/invoices/data')
RUN_NAME = 'qwen25-3b-international-v3-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
if WORK_DIR.name == 'notebooks' and (WORK_DIR.parent / 'data').is_dir():
    WORK_DIR = WORK_DIR.parent
RESULTS_DIR = WORK_DIR / 'results' / RUN_NAME
OUTPUT_DIR = RESULTS_DIR / 'checkpoints'
ADAPTER_DIR = RESULTS_DIR / 'lora_adapter'
EXPORT_GGUF = False  # optionnel : conversion coûteuse, après sauvegarde LoRA
DOWNLOAD_ARCHIVE = False  # Colab : True pour déclencher le téléchargement

SYSTEM_PROMPT = 'Tu es un extracteur de données. Réponds uniquement par un objet JSON valide.'
INSTRUCTION = (
    'Extrais les champs de la facture suivante au format JSON : supplier_name, '
    'invoice_date (AAAA-MM-JJ), total_ht, total_ttc, '
    'line_items (description, quantity, unit_price, total_ht).'
)

if RESULTS_DIR.exists():
    raise FileExistsError(f'Choisir un nouveau RUN_NAME : {RESULTS_DIR}')
print(f'Sortie du run : {RESULTS_DIR}')

if USE_WANDB:
    os.environ['WANDB_PROJECT'] = 'invoice-extraction-qlora'

## 2. Chargement des données

In [ ]:
def find_data_dir():
    if DATA_DIR_OVERRIDE is not None:
        candidate = Path(DATA_DIR_OVERRIDE).resolve()
        if not all((candidate / name).is_file() for name in ('train.jsonl', 'val.jsonl')):
            raise FileNotFoundError(f'train.jsonl et val.jsonl requis dans {candidate}')
        return candidate
    candidates = [WORK_DIR / 'data', WORK_DIR, Path('/content/data'), Path('/content')]
    if Path('/kaggle/input').is_dir():
        candidates.extend(path.parent for path in Path('/kaggle/input').rglob('train.jsonl'))
    matches = sorted({path.resolve() for path in candidates
                      if all((path / name).is_file() for name in ('train.jsonl', 'val.jsonl'))})
    if len(matches) != 1:
        raise ValueError(f'Un seul dossier de données requis, trouvé : {matches}. Régler DATA_DIR_OVERRIDE.')
    return matches[0]


def load_jsonl(path):
    with path.open(encoding='utf-8') as fh:
        return [json.loads(line) for line in fh if line.strip()]


DATA_DIR = find_data_dir()
train_rows = load_jsonl(DATA_DIR / 'train.jsonl')
val_rows = load_jsonl(DATA_DIR / 'val.jsonl')
for name, rows, expected_count, expected_region in [('train', train_rows, 480, 160), ('val', val_rows, 60, 20)]:
    if len(rows) != expected_count:
        raise ValueError(f'{name}: {len(rows)} lignes au lieu de {expected_count}')
    if Counter(row['locale'] for row in rows) != dict.fromkeys(('fr-FR', 'en-GB', 'en-US'), expected_region):
        raise ValueError(f'{name}: répartition régionale incorrecte')
    if any(row.get('dataset_version') != 'international-v3' for row in rows):
        raise ValueError(f'{name}: dataset international-v3 requis')
    if len({row['id'] for row in rows}) != len(rows) or len({row['input'] for row in rows}) != len(rows):
        raise ValueError(f'{name}: exemples dupliqués')
    for row in rows:
        if not isinstance(row['input'], str) or not row['input'].strip():
            raise ValueError(f'{name}: entrée vide ou invalide')
        if set(row['output']) != {'supplier_name', 'invoice_date', 'total_ht', 'total_ttc', 'line_items'}:
            raise ValueError(f'{name}: schéma cible incorrect')
if ({row['id'] for row in train_rows} & {row['id'] for row in val_rows}
        or {row['input'] for row in train_rows} & {row['input'] for row in val_rows}):
    raise ValueError('Fuite de données entre entraînement et validation')
DATA_HASHES = {name: hashlib.sha256((DATA_DIR / f'{name}.jsonl').read_bytes()).hexdigest()
               for name in ('train', 'val')}
print(f'Données v3 vérifiées : train={len(train_rows)}, val={len(val_rows)} | {DATA_DIR}')

## 3. Modèle 4 bits et adaptateurs LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU CUDA requis : activer le GPU dans Colab/Kaggle puis relancer.')
print(torch.cuda.get_device_name(0))

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)
model.print_trainable_parameters()

## 4. Formatage ChatML

In [ ]:
from datasets import Dataset


def to_messages(row):
    user_content = f"{INSTRUCTION}\n\n{row['input']}"
    target = json.dumps(row['output'], ensure_ascii=False)
    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_content},
        {'role': 'assistant', 'content': target},
    ]


def format_row(row):
    return {'text': tokenizer.apply_chat_template(to_messages(row), tokenize=False)}


train_ds = Dataset.from_list([format_row(r) for r in train_rows])
val_ds = Dataset.from_list([format_row(r) for r in val_rows])

TOKEN_LENGTHS = {}
for name, rows in [('train', train_rows), ('val', val_rows)]:
    lengths = [len(tokenizer.apply_chat_template(to_messages(row), tokenize=True)) for row in rows]
    TOKEN_LENGTHS[name] = {'min': min(lengths), 'max': max(lengths)}
    oversized = [(row['id'], length) for row, length in zip(rows, lengths) if length > MAX_SEQ_LENGTH]
    if oversized:
        raise ValueError(f'{name}: exemples trop longs {oversized[:10]}. Augmenter MAX_SEQ_LENGTH sans tronquer.')
print(f'Longueurs de séquence : {TOKEN_LENGTHS} (limite : {MAX_SEQ_LENGTH})')
print(train_ds[0]['text'][:600])

## 5. Entraînement avec SFTTrainer (TRL)

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported

RESULTS_DIR.mkdir(parents=True, exist_ok=False)
training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    dataset_text_field='text',
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    prediction_loss_only=True,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_steps=5,
    lr_scheduler_type='linear',
    optim='adamw_8bit',
    weight_decay=0.01,
    logging_steps=5,
    eval_strategy='steps',
    eval_steps=10,
    save_strategy='epoch',
    save_total_limit=2,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    seed=SEED,
    report_to='wandb' if USE_WANDB else 'none',
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=training_args,
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)

In [ ]:
trainer_stats = trainer.train()
model.save_pretrained(str(ADAPTER_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(ADAPTER_DIR))
for filename in ('adapter_config.json', 'adapter_model.safetensors', 'tokenizer_config.json'):
    artifact = ADAPTER_DIR / filename
    if not artifact.is_file() or artifact.stat().st_size == 0:
        raise RuntimeError(f'Sauvegarde absente ou vide : {artifact}')
print(f'Poids LoRA sauvegardés : {ADAPTER_DIR}')
eval_metrics = trainer.evaluate()
manifest = {
    'status': 'completed', 'base_model': MODEL_NAME,
    'base_model_revision': getattr(model.config, '_commit_hash', None),
    'dataset_version': 'international-v3', 'dataset_sha256': DATA_HASHES,
    'counts': {'train': len(train_rows), 'val': len(val_rows)},
    'training_args': training_args.to_dict(), 'token_lengths': TOKEN_LENGTHS,
    'train_metrics': trainer_stats.metrics, 'validation_metrics': eval_metrics,
    'weights': 'lora_adapter', 'requires_base_model': True,
    'gpu': torch.cuda.get_device_name(0),
    'packages': {name: version(name) for name in ('torch', 'unsloth', 'transformers', 'trl', 'peft', 'datasets')},
}
(RESULTS_DIR / 'training_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(trainer_stats.metrics)
print(eval_metrics)

## 6. Journal et courbes de perte

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_points = [(h['step'], h['loss']) for h in history if 'loss' in h]
eval_points = [(h['step'], h['eval_loss']) for h in history if 'eval_loss' in h]

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'train_log.json').write_text(json.dumps(history, indent=2), encoding='utf-8')

plt.figure(figsize=(8, 4))
if train_points:
    plt.plot(*zip(*train_points), label='train loss')
if eval_points:
    plt.plot(*zip(*eval_points), label='val loss')
plt.xlabel('step')
plt.ylabel('loss')
plt.legend()
plt.grid(True)
plt.savefig(RESULTS_DIR / 'loss_curve.png', dpi=150)
plt.show()

In [ ]:
archive_path = Path(shutil.make_archive(str(RESULTS_DIR), 'zip', root_dir=RESULTS_DIR.parent, base_dir=RESULTS_DIR.name))
print(f'Archive à récupérer avant arrêt du runtime : {archive_path}')
if DOWNLOAD_ARCHIVE:
    from google.colab import files
    files.download(str(archive_path))
else:
    print('Colab : panneau Fichiers > results > télécharger le ZIP. Kaggle : Save Version puis récupérer les Output files.')

## 7. Vérification rapide

In [ ]:
FastLanguageModel.for_inference(model)

sample = val_rows[0]
prompt_messages = [m for m in to_messages(sample) if m['role'] != 'assistant']
inputs = tokenizer.apply_chat_template(
    prompt_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors='pt',
    return_dict=True,
).to(model.device)

target_length = len(tokenizer(json.dumps(sample['output'], ensure_ascii=False), add_special_tokens=False)['input_ids'])
max_new_tokens = target_length + 32
prompt_length = inputs['input_ids'].shape[1]
if prompt_length + max_new_tokens > MAX_SEQ_LENGTH:
    raise ValueError('Contexte insuffisant pour la vérification : augmenter MAX_SEQ_LENGTH.')
outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
print('Prédit :', tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True))
print('Attendu :', json.dumps(sample['output'], ensure_ascii=False))

## 8. Export GGUF Q4_K_M (optionnel)

Les poids appris sont les adaptateurs LoRA déjà sauvegardés, à charger avec le modèle de base indiqué dans le manifeste. Le GGUF est une conversion fusionnée quantifiée pour llama.cpp, pas un second entraînement. Activer `EXPORT_GGUF` uniquement si les ressources et le temps le permettent. Après conversion, retélécharger la nouvelle archive.

In [ ]:
GGUF_DIR = RESULTS_DIR / 'gguf_q4_k_m'

if EXPORT_GGUF:
    model.save_pretrained_gguf(str(GGUF_DIR), tokenizer, quantization_method='q4_k_m')
    if not list(GGUF_DIR.glob('*.gguf')):
        raise RuntimeError('Export GGUF absent')
    archive_path = Path(shutil.make_archive(str(RESULTS_DIR), 'zip', root_dir=RESULTS_DIR.parent, base_dir=RESULTS_DIR.name))
    print(f'Archive mise à jour : {archive_path}')
    if DOWNLOAD_ARCHIVE:
        from google.colab import files
        files.download(str(archive_path))
else:
    print('Export GGUF désactivé ; adaptateurs LoRA déjà sauvegardés.')

In [ ]:
if HF_REPO_GGUF:
    model.push_to_hub_gguf(
        HF_REPO_GGUF,
        tokenizer,
        quantization_method='q4_k_m',
        token=os.environ['HF_TOKEN'],
    )
else:
    print('HF_REPO_GGUF non défini : upload ignoré.')

## 9. Inférence avec llama.cpp

Exécuter ces commandes dans un terminal, à partir de la racine du projet, après avoir téléchargé le fichier `.gguf` :

```bash
git clone https://github.com/ggml-org/llama.cpp
cmake -B llama.cpp/build -S llama.cpp
cmake --build llama.cpp/build --config Release -j
./llama.cpp/build/bin/llama-cli -m results/<RUN_NAME>/gguf_q4_k_m/<fichier>.gguf -n 2048 -p "Extrais les champs de la facture suivante au format JSON : ..."
```